<a href="https://colab.research.google.com/github/anamta-ansari/flyrank-ai/blob/main/work/notebooks/Week_03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from huggingface_hub import login
from google.colab import userdata

token = userdata.get("HF_TOKEN")

login(token)

print("Logged in successfully")

Logged in successfully


In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

dataset

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})

In [3]:
df = dataset["train"].select(range(100000)).to_pandas()
df.head(20)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0
5,2025-01-27,client_9958f0a7ae1df715,content_c782fa8abd4fce5e,True,True,True,False,21,0,1050.0,...,0,0,0,0,0,0,0,0,0,0
6,2025-01-27,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,True,True,True,False,13,0,127.0,...,0,0,0,0,0,0,0,0,0,0
7,2025-01-27,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,True,True,True,False,29,0,356.0,...,0,0,0,0,0,0,0,0,0,0
8,2025-01-27,client_9958f0a7ae1df715,content_e281674658070602,True,True,True,False,5,0,103.0,...,0,0,0,0,0,0,0,0,0,0
9,2025-01-27,client_9958f0a7ae1df715,content_658f53fa439c66ca,True,True,True,False,8,0,304.0,...,0,0,0,0,0,0,0,0,0,0


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The unit of analysis is a content-date observation.

Each row represents one content item identified by content_hash_id with its search performance and analytics signals recorded on a specific report_date.

The selected development window is February 2025 because it is a mid-panel historical period. This avoids using the final observed period and supports a past-to-future evaluation setup.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:

1. gsc_impressions
   Historical search visibility signal.

2. gsc_clicks
   Historical organic search traffic signal.

3. gsc_avg_position
   Historical ranking performance signal.

4. ga4_pageviews
   Historical user activity signal.

5. ga4_engaged_sessions
   Historical engagement quality signal.


Label:

A refresh opportunity proxy will be created from future performance changes.

The goal is to identify content items where historical signals indicate potential benefit from refreshing.


Context:

- report_date
- client_hash_id
- content_hash_id
- data availability indicators


Excluded:

Any future outcome-related metrics are excluded because they reveal the target information and create leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

This validation checks whether the defined unit of analysis is correct by verifying uniqueness of content_hash_id and report_date combinations.

In [4]:
grain_check = (
    df.groupby(
        ["content_hash_id", "report_date"]
    )
    .size()
)

grain_check.value_counts()

,count
1,100000


I measure the number of observations available during February 2025 and confirm the selected time window.

In [6]:
import pandas as pd
df["report_date"] = pd.to_datetime(df["report_date"])
feb_df = df[
    (df["report_date"] >= "2025-02-01") &
    (df["report_date"] <= "2025-02-28")
]

print("Rows:", len(feb_df))

feb_df["report_date"].min(), feb_df["report_date"].max()

Rows: 75985


(Timestamp('2025-02-01 00:00:00'), Timestamp('2025-02-28 00:00:00'))

The missing value analysis confirms that the selected feature set has strong availability.

Among the five features, only gsc_avg_position contains a missing observation. Since ranking position is only one missing value out of the dataset sample, the impact is expected to be minimal; however, missing handling should be considered before modeling.

In [7]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

df[features].isnull().sum()

,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,1
ga4_pageviews,0
ga4_engaged_sessions,0


The window verification shows that the dataset contains historical observations from 2025-01-27 to 2025-03-21, covering 53 days.

February 2025 is selected as the analysis window because it is a middle period within the available timeline and avoids using the final observed period.

In [8]:
window_check = {
    "first_observation": df["report_date"].min(),
    "last_observation": df["report_date"].max(),
    "number_of_days": (
        df["report_date"].max() -
        df["report_date"].min()
    ).days
}

window_check

{'first_observation': Timestamp('2025-01-27 00:00:00'),
 'last_observation': Timestamp('2025-03-21 00:00:00'),
 'number_of_days': 53}

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The dataset provides search and analytics performance signals, but it cannot fully explain why content performs well or poorly.

It cannot measure qualitative factors such as content quality, writing accuracy, user satisfaction, or competitor changes.

The data can help prioritize pages for refresh opportunities, but human review is still required before making content decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.